In [1]:
from mlops.datamodel.data_loader import DM_Loader
from mlops.model.convolutional.le_net import LeNetJaxImplementation
from structural.losses.categorical_crossentropy_lf import CCLossFunctionJax

dm_loader = DM_Loader()
dm_loader.set_new_data_from_config()
dataloader = dm_loader.generate_data_as_batch(32)

lenet_model = LeNetJaxImplementation()

iteration = next(dataloader)
observations = iteration['observations']
labels = iteration['labels']

lenet_model.construct(observations)
lenet_model.get_summary()

Using the kernel size (4, 4)
The number of trainable parameters is : 44426


detail_value
id layer_id   layer_name shape             trainable detail_key              
0  xm93XI5EmL C1         (None, 24, 24, 6) 156       kernel               150
                                                     bias                   6
1  zWukdDND1N S2         (None, 12, 12, 6) 0                                0
2  l5XfmCgJK1 C3         (None, 8, 8, 16)  2416      kernel              2400
                                                     bias                  16
3  F0eoe2ugo7 S4         (None, 4, 4, 16)  0                                0
4  MLCT0bwjsQ C5         (None, 120)       30840     kernel             30720
                                                     bias                 120
5  kE67UHjuR9 F6         (None, 84)        10164     weights            10080
                                                     bias                  84
6  DpQeDItwi1 Output     (None, 10)        850       weights              840
                                                     bias                  10

In [2]:
loss = CCLossFunctionJax(True)
loss.compute(
    predicted=lenet_model.forward(observations),
    truth=labels
)

Array(12.62515, dtype=float32)

In [3]:
grads, pure_weights, weight_headers = loss.compute_grad(lenet_model, observations, labels) # Good we have gradients

In [4]:
# The optimizer step 

# We use an SGD with momentum just like in the paper
from structural.optimizers.sgd_with_momentum import SGDWithMomentumJax 

optimizer = SGDWithMomentumJax(learning_rate=0.01, momentum=0.9)
new_weights = optimizer.step(grads, pure_weights, weight_headers)


In [6]:
lenet_model.set_weights(new_weights)

7

In [8]:
loss.compute(
    predicted=lenet_model.forward(observations),
    truth=labels
)

Array(12.2641535, dtype=float32)